# Notebook für Review "Historisiertes Gemeindeverzeichnis"

In [1]:
%pip install -q ipycytoscape networkx
import ipycytoscape as cy
import networkx as nx
import pandas as pd
from ext.sparql import query, display_result

Was ist der Unterschied zwischen Gemeinden (Identitäten und Versionen) unter ld.admin.ch/municipality und register.ld.admin.ch/agvch/municipality? --> Dazu noch was schreiben

## vl:Identity 

### Aktuelle Gemeinde (Bern)

In [2]:
df = await query("""

PREFIX vl: <https://version.link/>

SELECT * 
FROM <https://lindas.admin.ch/territorial>
WHERE {

    <https://ld.admin.ch/municipality/351> ?p ?o.

}


""", "https://test.lindas.admin.ch/query")

display_result(df)

,p,o
0,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,https://schema.ld.admin.ch/Municipality
1,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,https://schema.ld.admin.ch/PoliticalMunicipality
2,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,https://version.link/Identity
3,http://schema.org/name,Bern
4,http://schema.org/identifier,351
5,http://schema.org/inDefinedTermSet,https://ld.admin.ch/dimension/municipality
6,https://version.link/version,https://ld.admin.ch/municipality/version/15029
7,https://version.link/inVersionedIdentitySet,https://ld.admin.ch/municipality


Die Mandatory Properties aus version.link für eine Identität sind:

- OK: rdf:type vl:Identity
- OK: vl:version
- OK: vl:inVersionedIdentitySet
- OK: schema:identifier

Gemeinde unter register.ld.admin.ch/agvch

In [3]:
df = await query("""

PREFIX vl: <https://version.link/>

SELECT * 
FROM <https://lindas.admin.ch/territorial>
WHERE {

    <https://register.ld.admin.ch/agvch/municipality/351> ?p ?o.

}


""", "https://test.lindas.admin.ch/query")

display_result(df)

,p,o
0,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,https://ld.admin.ch/ech/71/Municipality
1,http://schema.org/identifier,351
2,https://ld.admin.ch/ech/71/municipalityId,351
3,https://ld.admin.ch/ech/71/municipalityEntryModeId,11


### Nicht mehr existierende Gemeinde (Ennenda)

In [4]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>

SELECT * 
FROM <https://lindas.admin.ch/territorial>
WHERE {

    <https://ld.admin.ch/municipality/1607> ?p ?o.
}


""", "https://test.lindas.admin.ch/query")

display_result(df)

,p,o
0,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,https://schema.ld.admin.ch/Municipality
1,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,https://schema.ld.admin.ch/PoliticalMunicipality
2,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,https://version.link/Identity
3,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,https://version.link/Deprecated
4,http://schema.org/name,Ennenda
5,http://schema.org/identifier,1607
6,http://schema.org/inDefinedTermSet,https://ld.admin.ch/dimension/municipality
7,https://version.link/version,https://ld.admin.ch/municipality/version/12279
8,https://version.link/inVersionedIdentitySet,https://ld.admin.ch/municipality


Die Mandatory Properties aus version.link für eine nicht mehr existierende Identität sind:

- OK: rdf:type vl:Identity
- OK: vl:version
- OK: vl:inVersionedIdentitySet
- OK: schema:identifier
- OK: vl:Deprecated

--> Hinweis: version.link lässt offen, dass man aktiven Gemeinden bspw. admin:Municipality und nicht mehr aktiven admin:AbolishedMunicipality zuweist (und kein admin:Municipality mehr), somit könnte man einfacher nach allen aktiven suchen. Der Grund wieso ein vl:Deprecated eingeführt wurde, ist vor allem für die Versionen: Das Prinzip dort ist, dass man nichts ändern will, was man mal hinzugefügt hat, man will also nicht, dass bspw. ein vl:ActiveVersion durch ein vl:DeprecatedVersion ersetzt wird, dieser Graph ist "strictly growing", man muss also ein vl:Deprecated hinzutun, ohne etwas wegzunehmen. Damit bei den Identities der gleiche Mechanismus spielt, ist es dort auch so, aber in der Praxis könnte man immer zusätzlich zu vl:Identity als Typ auch etwas setzen, das man dann wenn die Identität abolished wird, weglässt und durch ein Spezifikum ersetzt, also bspw. admin:PoliticalMunicipality durch admin:AbolishedPoliticalMunicipality

--> Frage: Für was ist das schema:DefinedTermSet https://ld.admin.ch/dimension/municipality notwendig? Wohl für visualize.admin.ch - aber gehören da vl:Deprecated auch rein?

### Anzahl aller Gemeinden

In [5]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>

SELECT (COUNT(*) AS ?count)
FROM <https://lindas.admin.ch/territorial>
WHERE {

    ?muni a admin:PoliticalMunicipality.

}
""", "https://test.lindas.admin.ch/query")

display_result(df)

,count
0,3292


### Anzahl aller nicht mehr aktiven Gemeinden

In [6]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>

SELECT (COUNT(*) AS ?count)
FROM <https://lindas.admin.ch/territorial>
WHERE {

    ?muni a admin:PoliticalMunicipality;
        a vl:Deprecated.

}
""", "https://test.lindas.admin.ch/query")

display_result(df)

,count
0,1155


### Anzahl aller aktiver Gemeinden

In [7]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>

SELECT (COUNT(*) AS ?count)
FROM <https://lindas.admin.ch/territorial>
WHERE {

    ?muni a admin:PoliticalMunicipality;
    
    FILTER NOT EXISTS {?muni a vl:Deprecated}
    #MINUS {?muni a vl:Deprecated}

}
""", "https://test.lindas.admin.ch/query")

display_result(df)

,count
0,2137


## vl:Version

### ld.admin.ch/municipality/version

In [8]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>

SELECT * 
FROM <https://lindas.admin.ch/territorial>
WHERE {

    <https://ld.admin.ch/municipality/version/12279> ?p ?o.
}


""", "https://test.lindas.admin.ch/query")

display_result(df)

,p,o
0,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://schema.org/DefinedTerm
1,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,https://version.link/Version
2,http://schema.org/name,Ennenda
3,http://schema.org/identifier,12279
4,http://schema.org/isPartOf,https://ld.admin.ch/district/version/10109
5,http://schema.org/validFrom,1960-01-01
6,http://www.geonames.org/ontology#featureCode,http://www.geonames.org/ontology#A.ADM3
7,http://schema.org/validThrough,2010-12-31
8,http://schema.org/endDate,2010-12-31
9,http://schema.org/legalName,Ennenda


Die Mandatory Properties aus version.link für eine Version sind:

- OK: rdf:type vl:Version
- OK: vl:identity
- OK: vl:identityIdentifier
- **NOK**: vl:inVersionedIdentitySet
- OK: schema:identifier
- OK: vl:successor

### register.ld.admin.ch/agvch/municipalityversion

In [9]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>

SELECT * 
FROM <https://lindas.admin.ch/territorial>
WHERE {

    <https://register.ld.admin.ch/agvch/municipalityversion/12279> ?p ?o.
}


""", "https://test.lindas.admin.ch/query")

display_result(df)

,p,o
0,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,https://ld.admin.ch/ech/71/MunicipalityVersion
1,http://schema.org/name,Ennenda
2,http://schema.org/identifier,12279
3,http://schema.org/legalName,Ennenda
4,https://ld.admin.ch/ech/71/cantonAbbreviation,GL
5,https://ld.admin.ch/ech/71/shortName,Ennenda
6,https://ld.admin.ch/ech/71/districtHistId,10109
7,https://ld.admin.ch/ech/71/municipalityId,1607
8,https://ld.admin.ch/ech/71/municipalityStatusId,1
9,https://ld.admin.ch/ech/71/municipalityAdmissionNumber,1000


## Veränderungsprozesse

Welche Gemeinden spielten alle eine Rolle bei der heutigen Gemeinde "Klosters" (BFS Nummer 3871)?

In [10]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?participants ?name
FROM <https://lindas.admin.ch/territorial>
WHERE {

    <https://ld.admin.ch/municipality/3871> vl:version ?final.
    ?participants vl:successor* ?final;
        schema:name ?name.
    
    MINUS {?participants a vl:ChangeEvent}
}


""", "https://test.lindas.admin.ch/query")

display_result(df)

,participants,name
0,https://ld.admin.ch/municipality/version/10155,Saas
1,https://ld.admin.ch/municipality/version/11299,Klosters
2,https://ld.admin.ch/municipality/version/13233,Klosters-Serneus
3,https://ld.admin.ch/municipality/version/14310,Klosters-Serneus
4,https://ld.admin.ch/municipality/version/14313,Saas
5,https://ld.admin.ch/municipality/version/16070,Klosters-Serneus
6,https://ld.admin.ch/municipality/version/16610,Klosters
7,https://ld.admin.ch/municipality/version/15682,Klosters-Serneus


Grafische Darstellung

In [11]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?source ?target
FROM <https://lindas.admin.ch/territorial>
WHERE {

    <https://ld.admin.ch/municipality/3871> vl:version ?final.
    ?source vl:successor* ?final;
        vl:successor ?target.
    
    MINUS {?source a vl:ChangeEvent}
}


""", "https://test.lindas.admin.ch/query")


G = nx.from_pandas_edgelist(df, create_using=nx.DiGraph())

my_style = [
    {
        'selector': 'node',
         'style': {
            'font-family': 'helvetica',
            'font-size': '8px',
            'label': 'data(id)'
         }
    },
    {
        "selector": "edge.directed",
        "style": {
            "curve-style": "bezier",
            "target-arrow-shape": "triangle",
        }
    }
]

muni = cy.CytoscapeWidget()
muni.graph.add_graph_from_networkx(G, directed=True)
muni.set_style(my_style)
muni.set_layout(name = "dagre", rankDir = "LR")
muni

CytoscapeWidget(cytoscape_layout={'name': 'dagre', 'rankDir': 'LR'}, cytoscape_style=[{'selector': 'node', 'st…

Für Bolligen (BFS Nummer 352) als Beispiel für eine Gemeinde, die sich geteilt hat:

In [12]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?source ?target
FROM <https://lindas.admin.ch/territorial>
WHERE {

    <https://ld.admin.ch/municipality/352> vl:version ?final.
    ?source vl:successor* ?final;
        vl:successor ?target;
        schema:name ?sourceName.
    ?target schema:name ?targetName.
    
    MINUS {?source a vl:ChangeEvent}
}


""", "https://test.lindas.admin.ch/query")


G = nx.from_pandas_edgelist(df, source="source", target="target", create_using=nx.DiGraph())

my_style = [
    {
        'selector': 'node',
         'style': {
            'font-family': 'helvetica',
            'font-size': '8px',
            'label': 'data(id)'
         }
    },
    {
        "selector": "edge.directed",
        "style": {
            "curve-style": "bezier",
            "target-arrow-shape": "triangle",
        }
    }
]

muni = cy.CytoscapeWidget()
muni.graph.add_graph_from_networkx(G, directed=True)
muni.set_style(my_style)
muni.set_layout(name = "dagre", rankDir = "LR")
muni

CytoscapeWidget(cytoscape_layout={'name': 'dagre', 'rankDir': 'LR'}, cytoscape_style=[{'selector': 'node', 'st…

todo: Nodes mit Namen anschreiben (aber die Nodes müssen noch irgendwie eine individuelle ID haben, sonst klappt es nicht, weil die Namen häufig die gleichen sind)

### Alle Gemeinden, die sich geteilt haben

In [13]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT * 
FROM <https://lindas.admin.ch/territorial>
WHERE {
    
    ?source schema:name ?name
    
    {
        SELECT ?source (COUNT(?target) as ?successors)
        WHERE {

            ?source vl:successor ?target;

            MINUS {?source a vl:ChangeEvent}
        } 
        GROUP BY ?source
    }
    FILTER(?successors > 1)
}

""", "https://test.lindas.admin.ch/query")

display_result(df)

,source,name,successors
0,https://ld.admin.ch/municipality/version/10790,Maienfeld,2
1,https://ld.admin.ch/municipality/version/10792,Fläsch,2
2,https://ld.admin.ch/municipality/version/12036,Gottshaus,2
3,https://ld.admin.ch/municipality/version/12346,Monte,2
4,https://ld.admin.ch/municipality/version/11252,G.-geb. Maienfeld-Fläsch,2
5,https://ld.admin.ch/municipality/version/12481,Opfershofen (TG),2
6,https://ld.admin.ch/municipality/version/11296,Bolligen,3
7,https://ld.admin.ch/municipality/version/12538,Neftenbach,2
8,https://ld.admin.ch/municipality/version/11320,Arni-Islisberg,2
9,https://ld.admin.ch/municipality/version/12612,Lavertezzo,3


### Analyse Berg (TG)

Dies ist ein Beispiel für eine Abfrage, wo zu Beginn oder am Ende nicht eine einzelne Version steht, sondern man von einer Version in der "Mitte" des Geschehens ausgeht.

In [14]:
df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT ?source ?target ?sourceName ?targetName
FROM <https://lindas.admin.ch/territorial>
WHERE {
    
    {
        <https://ld.admin.ch/municipality/version/14062> vl:predecessor* ?target.
        ?target vl:predecessor ?source;
            schema:name ?targetName.
        ?source schema:name ?sourceName.
    }
    UNION
    {
        <https://ld.admin.ch/municipality/version/14062> vl:successor* ?source.
        ?source vl:successor ?target;
            schema:name ?sourceName.
        ?target schema:name ?targetName.
    }
}

""", "https://test.lindas.admin.ch/query")

display_result(df)

G = nx.from_pandas_edgelist(df, source="source", target="target", create_using=nx.DiGraph())

my_style = [
    {
        'selector': 'node',
         'style': {
            'font-family': 'helvetica',
            'font-size': '8px',
            'label': 'data(id)'
         }
    },
    {
        "selector": "edge.directed",
        "style": {
            "curve-style": "bezier",
            "target-arrow-shape": "triangle",
        }
    }
]

muni = cy.CytoscapeWidget()
muni.graph.add_graph_from_networkx(G, directed=True)
muni.set_style(my_style)
muni.set_layout(name = "dagre", rankDir = "LR")
muni

,source,target,sourceName,targetName
0,https://ld.admin.ch/municipality/version/11930,https://ld.admin.ch/municipality/version/14062,Guntershausen b. Birwin.,Berg (TG)
1,https://ld.admin.ch/municipality/version/12041,https://ld.admin.ch/municipality/version/14062,Graltshausen,Berg (TG)
2,https://ld.admin.ch/municipality/version/12401,https://ld.admin.ch/municipality/version/14062,Mauren,Berg (TG)
3,https://ld.admin.ch/municipality/version/11761,https://ld.admin.ch/municipality/version/13753,Andhausen,Berg (TG)
4,https://ld.admin.ch/municipality/version/11877,https://ld.admin.ch/municipality/version/13753,Berg (TG),Berg (TG)
5,https://ld.admin.ch/municipality/version/13753,https://ld.admin.ch/municipality/version/14062,Berg (TG),Berg (TG)
6,https://ld.admin.ch/municipality/version/14062,https://ld.admin.ch/municipality/version/14063,Berg (TG),Berg (TG)
7,https://ld.admin.ch/municipality/version/14062,https://ld.admin.ch/municipality/version/14064,Berg (TG),Weinfelden
8,https://ld.admin.ch/municipality/version/14063,https://ld.admin.ch/municipality/version/14111,Berg (TG),Lengwil
9,https://ld.admin.ch/municipality/version/14063,https://ld.admin.ch/municipality/version/14112,Berg (TG),Berg (TG)


CytoscapeWidget(cytoscape_layout={'name': 'dagre', 'rankDir': 'LR'}, cytoscape_style=[{'selector': 'node', 'st…